# 09 — Base analítica corrigida (Tuberculose, SINAN 2014–2024)

Este notebook reconstrói a base de análise a partir dos arquivos brutos `TUBEBR14–24.csv` (Brasil inteiro),
aplicando as correções definidas no plano aprovado. Ele substitui, para os notebooks 10–15, os arquivos
`tuberculose_ilheus_itabuna_final.csv` e `tuberculose_10_cidades_final.csv` (que continuam na pasta, intactos).

**Correções aplicadas**

1. **Uma definição de cidade por pergunta:** guardamos município de residência, de notificação e de tratamento atual.
   Incidência e perfil usam **residência**; desempenho do serviço usa **notificação** ou **tratamento atual**.
2. **Rótulos de desfecho** padronizados em um único dicionário (resolve o caso "Transferencia" x "Transferência").
3. **Códigos 0, 9 e brancos** viram `NaN` (ignorado) nas variáveis sim/não; nunca entram no denominador.
4. **Coorte de encerramento** no padrão do Ministério da Saúde: casos novos, sem "mudança de diagnóstico".
5. **População anual** (Censos 2010 e 2022 + estimativa IBGE 2024) em vez de população fixa.
6. **Possíveis duplicidades** marcadas (não removidas) para análise de sensibilidade.
7. **Vitória da Conquista** com o código correto (`293330`; o notebook 07 usava `291950`).
8. **Caminhos relativos** à pasta do projeto.

Fontes externas (salvas em `data/external/`):

| Arquivo | Conteúdo | Fonte |
| --- | --- | --- |
| `censo2010.csv` | População do Censo 2010 e área (km²) | IBGE, via repositório `turicas/rows` |
| `censo2022.csv` | População do Censo 2022 | IBGE, via repositório `goldenluke/labsus` |
| `populacao-AAAA_*.csv` | Estimativas anuais do IBGE (TCU) | IBGE, via repositório `turicas/censo-ibge` |
| `IDHMunic2010.csv` | IDH-M 2010 | Atlas do Desenvolvimento Humano (PNUD/IPEA/FJP) |
| `municipiosAreas_ibge2018.csv` | Área territorial 2018 | IBGE, via `datasets-br/city-codes` |
| `municipios.csv` | Códigos IBGE, nome, UF, capital | `kelvins/municipios-brasileiros` |
| `geojs-29-mun.json` | Malha municipal da Bahia | `tbrugz/geodata-br` |

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path('..', 'src').resolve()))

import json
import unicodedata
import numpy as np
import pandas as pd
import tb_utils as tb

pd.set_option('display.max_columns', 60)
print('Arquivos brutos encontrados:')
for a in tb.arquivos_brutos():
    print(' -', Path(a).name)

Arquivos brutos encontrados:
 - TUBEBR14.csv
 - TUBEBR15.csv
 - TUBEBR16.csv
 - TUBEBR17.csv
 - TUBEBR18.csv
 - TUBEBR19.csv
 - TUBEBR20.csv
 - TUBEBR21.csv
 - TUBEBR22.csv
 - TUBEBR23.csv
 - TUBEBR24.csv


## 1. População anual por município

As estimativas anuais do IBGE de 2014 a 2021 foram projetadas a partir do Censo 2010 e ficaram muito
distantes do Censo 2022 em vários municípios. Exemplo: a estimativa de 2021 para Itabuna era 214 mil
habitantes, e o Censo 2022 contou 186,7 mil. Usar essas estimativas cria um "salto" artificial nas taxas
em 2022.

**Escolha:** usamos a **taxa geométrica de crescimento entre os Censos 2010 e 2022** para todos os anos:
interpolação em 2014–2021, o próprio Censo em 2022 e extrapolação com a mesma taxa em 2023–2024 (o IBGE não
publicou estimativa para 2023, e a de 2024 incorpora um ajuste de cobertura que cria um novo salto: Ilhéus
passaria de 178,6 mil em 2022 para 189 mil em 2024). As estimativas originais ficam numa coluna à parte para
análise de sensibilidade.

In [2]:
ext = tb.EXTERNAL

def normaliza_nome(s):
    s = unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode().upper()
    return ''.join(ch for ch in s if ch.isalnum())

municipios = pd.read_csv(ext / 'municipios.csv')
uf_por_codigo = {11: 'RO', 12: 'AC', 13: 'AM', 14: 'RR', 15: 'PA', 16: 'AP', 17: 'TO', 21: 'MA', 22: 'PI', 23: 'CE',
                 24: 'RN', 25: 'PB', 26: 'PE', 27: 'AL', 28: 'SE', 29: 'BA', 31: 'MG', 32: 'ES', 33: 'RJ', 35: 'SP',
                 41: 'PR', 42: 'SC', 43: 'RS', 50: 'MS', 51: 'MT', 52: 'GO', 53: 'DF'}
municipios['uf'] = municipios['codigo_uf'].map(uf_por_codigo)
municipios['cod7'] = municipios['codigo_ibge'].astype(str)
municipios['cod6'] = municipios['cod7'].str[:6]
municipios['chave'] = municipios['uf'] + '|' + municipios['nome'].map(normaliza_nome)

c2010 = pd.read_csv(ext / 'censo2010.csv')
c2010['chave'] = c2010['state'] + '|' + c2010['city'].map(normaliza_nome)
c2010 = c2010.merge(municipios[['chave', 'cod6']], on='chave', how='left')
print(f"Censo 2010: {len(c2010)} municípios, {c2010['cod6'].isna().sum()} sem correspondência de nome (mudaram de nome)")
c2010 = c2010.dropna(subset=['cod6'])[['cod6', 'inhabitants']].rename(columns={'inhabitants': 'pop2010'})

c2022 = pd.read_csv(ext / 'censo2022.csv', sep=';', dtype={'cod_mun_ibge_6': str}, encoding='utf-8-sig')
c2022 = c2022.rename(columns={'cod_mun_ibge_6': 'cod6', 'populacao': 'pop2022'})[['cod6', 'pop2022']]

estimativas = {}
for arq in sorted(ext.glob('populacao-20*.csv')):
    ano = int(arq.name[10:14])
    if ano in (2022, 2025):  # 2022 = prévia (usamos o Censo); 2025 fora do período
        continue
    e = pd.read_csv(arq, dtype={'codigo_municipio': str})
    estimativas[ano] = e.assign(cod6=e['codigo_municipio'].str[:6]).set_index('cod6')['populacao']
est = pd.DataFrame(estimativas)

base = c2022.merge(c2010, on='cod6', how='left').set_index('cod6')
linhas = []
for ano in tb.ANOS:
    if ano == 2022:
        pop, metodo = base['pop2022'], 'Censo 2022'
    else:
        fator = (ano - 2010) / 12
        pop = base['pop2010'] * (base['pop2022'] / base['pop2010']) ** fator
        pop = pop.fillna(base['pop2022'])  # municípios sem Censo 2010 casado: usa o Censo 2022
        metodo = 'interpolação geométrica Censos 2010-2022' if ano < 2022 else 'extrapolação da taxa 2010-2022'
    linhas.append(pd.DataFrame({'cod6': base.index, 'ano': ano, 'populacao': pop.round().values,
                                'populacao_estimativa_ibge': est[ano].reindex(base.index).values if ano in est else np.nan,
                                'metodo': metodo}))
pop = pd.concat(linhas, ignore_index=True)
pop.to_csv(ext / 'populacao_municipios_2014_2024.csv', index=False)

(pop[pop['cod6'].isin([tb.COD_ILHEUS, tb.COD_ITABUNA])]
   .assign(cidade=lambda d: d['cod6'].map(tb.CIDADES))
   .pivot(index='ano', columns='cidade', values=['populacao', 'populacao_estimativa_ibge'])
   .astype('Int64'))

Censo 2010: 5565 municípios, 13 sem correspondência de nome (mudaram de nome)


populacao         populacao_estimativa_ibge        
cidade    Ilhéus Itabuna                    Ilhéus Itabuna
ano                                                       
2014      182355  198497                    182350  218925
2015      181887  196983                    180213  219680
2016      181421  195481                    178210  220386
2017      180956  193991                    176341  221046
2018      180492  192512                    164844  212740
2019      180030  191044                    162327  213223
2020      179568  189588                    159923  213685
2021      179108  188142                    157639  214123
2022      178649  186708                      <NA>    <NA>
2023      178191  185285                      <NA>    <NA>
2024      177734  183872                    189028  196676

A tabela mostra o problema: pela estimativa antiga, Ilhéus "perde" 25 mil habitantes entre 2014 e 2021 e
"ganha" 21 mil em 2022, e Itabuna "perde" 27 mil de uma vez. A curva entre os censos remove esses saltos.

## 2. Indicadores municipais (IDH-M, área, densidade)

In [3]:
idh = pd.read_csv(ext / 'IDHMunic2010.csv', sep=';', skiprows=1, encoding='utf-8-sig', dtype={'Código': str})
idh = idh.rename(columns={'Código': 'cod7', '2010': 'idhm_2010'})[['cod7', 'idhm_2010']]
idh['idhm_2010'] = pd.to_numeric(idh['idhm_2010'].astype(str).str.replace(',', '.'), errors='coerce')
idh['cod6'] = idh['cod7'].str[:6]

area = pd.read_csv(ext / 'municipiosAreas_ibge2018.csv', dtype={'cod_ibge': str})
area['cod6'] = area['cod_ibge'].str[:6]

indic = (municipios[['cod6', 'nome', 'uf', 'capital', 'latitude', 'longitude']]
         .merge(idh[['cod6', 'idhm_2010']], on='cod6', how='left')
         .merge(area[['cod6', 'area_km2']], on='cod6', how='left')
         .merge(c2022, on='cod6', how='left'))
indic['densidade_2022'] = indic['pop2022'] / indic['area_km2']
indic.to_csv(ext / 'indicadores_municipios.csv', index=False)
indic[indic['cod6'].isin(tb.CIDADES)].set_index('nome')[['uf', 'idhm_2010', 'area_km2', 'pop2022', 'densidade_2022']].round(3)

,uf,idhm_2010,area_km2,pop2022,densidade_2022
nome,,,,,
Aracaju,SE,0.770,182.0,602757.0,3311.852
Curitiba,PR,0.823,435.0,1773718.0,4077.513
Feira de Santana,BA,0.712,1304.0,616272.0,472.601
Ilhéus,BA,0.690,1589.0,178649.0,112.429
Itabuna,BA,0.712,401.0,186708.0,465.606
Recife,PE,0.772,219.0,1488920.0,6798.721
Rio de Janeiro,RJ,0.799,1200.0,6211223.0,5176.019
Salvador,BA,0.759,694.0,2417678.0,3483.686
São Paulo,SP,0.805,1521.0,11451999.0,7529.256


## 3. Leitura dos dados brutos

Lemos os 11 arquivos do Brasil. Guardamos com todos os detalhes os registros ligados à Bahia (notificação,
residência ou tratamento) e às 10 cidades do estudo. Para o Brasil inteiro guardamos uma tabela agregada
(casos por ano, município de residência e tipo de entrada), usada para taxas de referência.

In [4]:
COLUNAS = ['NU_ANO', 'DT_NOTIFIC', 'DT_DIAG', 'ANO_NASC', 'NU_IDADE_N', 'CS_SEXO', 'CS_GESTANT', 'CS_RACA',
           'CS_ESCOL_N', 'SG_UF_NOT', 'ID_MUNICIP', 'SG_UF', 'ID_MN_RESI', 'DT_DIGITA', 'TRATAMENTO',
           'RAIOX_TORA', 'FORMA', 'EXTRAPU1_N', 'AGRAVAIDS', 'AGRAVALCOO', 'AGRAVDIABE', 'AGRAVDOENC',
           'AGRAVOUTRA', 'AGRAVDROGA', 'AGRAVTABAC', 'BACILOSC_E', 'CULTURA_ES', 'HIV', 'HISTOPATOL',
           'DT_INIC_TR', 'TRAT_SUPER', 'TRATSUP_AT', 'NU_CONTATO', 'NU_COMU_EX', 'SG_UF_AT', 'ID_MUNIC_A',
           'SITUA_ENCE', 'DT_ENCERRA', 'TPUNINOT', 'POP_LIBER', 'POP_RUA', 'POP_SAUDE', 'POP_IMIG',
           'BENEF_GOV', 'TEST_MOLEC', 'TEST_SENSI', 'ANT_RETRO', 'TRANSF']

def le_arquivo(caminho):
    for enc in ('utf-8', 'latin1'):
        try:
            return pd.read_csv(caminho, usecols=lambda c: c.strip() in COLUNAS, dtype=str, encoding=enc,
                               on_bad_lines='skip')
        except UnicodeDecodeError:
            continue

detalhe, agregado, volumetria = [], [], []
for arq in tb.arquivos_brutos():
    d = le_arquivo(arq)
    d.columns = d.columns.str.strip()
    for c in ['SG_UF_NOT', 'ID_MUNICIP', 'SG_UF', 'ID_MN_RESI', 'ID_MUNIC_A', 'TRATAMENTO', 'DT_DIAG', 'NU_ANO']:
        d[c] = tb.limpa(d[c])
    d['ano_diag'] = pd.to_numeric(d['DT_DIAG'].str[:4], errors='coerce').fillna(pd.to_numeric(d['NU_ANO'], errors='coerce'))
    volumetria.append({'arquivo': Path(arq).name, 'registros': len(d), 'Bahia (notificação)': int((d['SG_UF_NOT'] == '29').sum())})
    agregado.append(d.groupby(['ano_diag', 'SG_UF', 'ID_MN_RESI', 'TRATAMENTO'], dropna=False).size().rename('casos').reset_index())
    manter = ((d['SG_UF_NOT'] == '29') | (d['SG_UF'] == '29') | (d['SG_UF_AT'].str.strip() == '29')
              | d['ID_MN_RESI'].isin(tb.CIDADES) | d['ID_MUNICIP'].isin(tb.CIDADES) | d['ID_MUNIC_A'].isin(tb.CIDADES))
    detalhe.append(d[manter].assign(arquivo=Path(arq).name))

raw = pd.concat(detalhe, ignore_index=True)
agregado = pd.concat(agregado, ignore_index=True)
pd.DataFrame(volumetria)

,arquivo,registros,Bahia (notificação)
0,TUBEBR14.csv,85216,5612
1,TUBEBR15.csv,85462,5304
2,TUBEBR16.csv,86210,5300
3,TUBEBR17.csv,90295,5486
4,TUBEBR18.csv,94735,5670
5,TUBEBR19.csv,95849,5635
6,TUBEBR20.csv,85962,4676
7,TUBEBR21.csv,91310,5059
8,TUBEBR22.csv,103330,5514
9,TUBEBR23.csv,109854,5535


## 4. Decodificação e variáveis derivadas

In [5]:
df = pd.DataFrame(index=raw.index)
df['arquivo'] = raw['arquivo']
for col in ['DT_NOTIFIC', 'DT_DIAG', 'DT_INIC_TR', 'DT_ENCERRA', 'DT_DIGITA']:
    df[col.lower()] = pd.to_datetime(tb.limpa(raw[col]), format='%Y%m%d', errors='coerce')
df['ano_diag'] = raw['ano_diag'].astype('Int64')
df['ano_notificacao'] = pd.to_numeric(raw['NU_ANO'], errors='coerce').astype('Int64')
df['mes_diag'] = df['dt_diag'].dt.to_period('M').astype(str)

# Municípios
df['uf_not'] = raw['SG_UF_NOT']
df['mun_not'] = raw['ID_MUNICIP']
df['uf_res'] = raw['SG_UF']
df['mun_res'] = raw['ID_MN_RESI']
df['mun_trat'] = raw['ID_MUNIC_A']
for c in ['mun_not', 'mun_res', 'mun_trat']:
    df[c.replace('mun', 'cidade')] = df[c].map(tb.CIDADES)
nomes = municipios.set_index('cod6')['nome']
df['nome_mun_res'] = df['mun_res'].map(nomes)
df['nome_mun_not'] = df['mun_not'].map(nomes)

# Pessoa
idade = tb.limpa(raw['NU_IDADE_N'])
unidade, valor = idade.str[0], pd.to_numeric(idade.str[1:], errors='coerce')
df['idade'] = np.where(unidade == '4', valor, np.where(unidade.isin(['1', '2', '3']), 0, np.nan))
df.loc[df['idade'] > 110, 'idade'] = np.nan
cortes = list(range(0, 85, 5)) + [200]
rotulos = [f'{i}-{i+4}' for i in range(0, 80, 5)] + ['80+']
df['faixa_etaria'] = pd.cut(df['idade'], bins=cortes, labels=rotulos, right=False)
df['faixa_etaria_ampla'] = pd.cut(df['idade'], bins=[0, 15, 30, 45, 60, 200],
                                  labels=['0-14', '15-29', '30-44', '45-59', '60+'], right=False)
df['sexo'] = tb.limpa(raw['CS_SEXO']).map(tb.MAPA_SEXO)
df['raca'] = tb.sem_zero_esquerda(raw['CS_RACA']).map(tb.MAPA_RACA)
df['raca_negra'] = df['raca'].map({'Preta': 1.0, 'Parda': 1.0, 'Branca': 0.0, 'Amarela': 0.0, 'Indígena': 0.0})
esc = tb.sem_zero_esquerda(raw['CS_ESCOL_N'])
df['escolaridade'] = esc.map(tb.MAPA_ESCOL)
df['escolaridade_grupo'] = esc.map(tb.ESCOL_GRUPO)

# Caso
df['tipo_entrada'] = tb.sem_zero_esquerda(raw['TRATAMENTO']).map(tb.MAPA_ENTRADA)
df['forma'] = tb.sem_zero_esquerda(raw['FORMA']).map(tb.MAPA_FORMA)
df['pulmonar'] = df['forma'].isin(['Pulmonar', 'Pulmonar + extrapulmonar'])
df['baciloscopia'] = tb.sem_zero_esquerda(raw['BACILOSC_E']).map(tb.MAPA_BACILO)
df['cultura'] = tb.sem_zero_esquerda(raw['CULTURA_ES']).map(tb.MAPA_CULTURA)
df['trm'] = tb.sem_zero_esquerda(raw['TEST_MOLEC']).map(tb.MAPA_TRM)
df['hiv'] = tb.sem_zero_esquerda(raw['HIV']).map(tb.MAPA_HIV)
df['hiv_testado'] = df['hiv'].map({'Positivo': 1.0, 'Negativo': 1.0, 'Em andamento': 1.0, 'Não realizado': 0.0})
df['cultura_realizada'] = df['cultura'].map({'Positiva': 1.0, 'Negativa': 1.0, 'Em andamento': 1.0, 'Não realizada': 0.0})
df['trm_realizado'] = df['trm'].map(lambda v: np.nan if pd.isna(v) else float(v != 'Não realizado'))
df['confirmacao_lab'] = ((df['baciloscopia'] == 'Positiva') | (df['cultura'] == 'Positiva')
                         | df['trm'].isin(['Detectável sensível à rifampicina', 'Detectável resistente à rifampicina'])).astype(float)
df['bacilifero'] = (df['pulmonar'] & (df['confirmacao_lab'] == 1)).astype(float)
df['tipo_unidade'] = tb.sem_zero_esquerda(raw['TPUNINOT']).map(tb.MAPA_TIPO_UNIDADE).fillna('Outros / não informado')

for orig, novo in tb.COMORBIDADES.items():
    df[novo] = tb.limpa(raw[orig]).map(tb.SIM_NAO)
df['tdo_2014_campo_antigo'] = tb.limpa(raw['TRAT_SUPER']).map(tb.SIM_NAO)

df['n_contatos'] = pd.to_numeric(raw['NU_CONTATO'], errors='coerce')
df['n_contatos_examinados'] = pd.to_numeric(raw['NU_COMU_EX'], errors='coerce')
df.loc[df['n_contatos'] > 100, 'n_contatos'] = np.nan
df.loc[df['n_contatos_examinados'] > df['n_contatos'], 'n_contatos_examinados'] = np.nan

# Desfecho
cod_desf = tb.sem_zero_esquerda(raw['SITUA_ENCE'])
df['desfecho'] = cod_desf.map(tb.MAPA_DESFECHO).fillna('Sem encerramento')
grupo = {'Cura': 'Cura', 'Abandono': 'Abandono', 'Abandono primário': 'Abandono', 'Óbito por TB': 'Óbito por TB',
         'Óbito por outras causas': 'Óbito por outras causas', 'Transferência': 'Transferência',
         'Mudança de diagnóstico': 'Mudança de diagnóstico', 'Sem encerramento': 'Sem encerramento'}
df['desfecho_grupo'] = df['desfecho'].map(grupo).fillna('TB-DR / falência / mudança de esquema')
df['cura'] = df['desfecho'].eq('Cura').astype(float)
df['abandono'] = df['desfecho'].isin(['Abandono', 'Abandono primário']).astype(float)
df['obito_tb'] = df['desfecho'].eq('Óbito por TB').astype(float)
# Desfecho desfavorável (para a regressão): abandono, abandono primário, óbito por TB ou falência x cura
df['desfavoravel'] = np.select([df['desfecho'].eq('Cura'),
                                df['desfecho'].isin(['Abandono', 'Abandono primário', 'Óbito por TB', 'Falência'])],
                               [0.0, 1.0], default=np.nan)

# Tempos
df['dias_diag_trat'] = (df['dt_inic_tr'] - df['dt_diag']).dt.days
df['data_trat_invalida'] = df['dias_diag_trat'].notna() & ((df['dias_diag_trat'] < 0) | (df['dias_diag_trat'] > 365))
df.loc[df['data_trat_invalida'], 'dias_diag_trat'] = np.nan
df['dias_diag_notif'] = (df['dt_notific'] - df['dt_diag']).dt.days
df['dias_ate_encerramento'] = (df['dt_encerra'] - df['dt_diag']).dt.days
df['erro_cronologico'] = (df['dias_ate_encerramento'] < 0) | (df['dias_ate_encerramento'] > 1825)

faixas_espera = pd.cut(df['dias_diag_trat'], bins=[-1, 0, 7, 15, 30, 365],
                       labels=['Mesmo dia', '1-7 dias', '8-15 dias', '16-30 dias', 'Mais de 30 dias'])
df['espera_tratamento'] = faixas_espera

df.shape

(287958, 67)

## 5. Qualidade: duplicidades e inconsistências

Em TB, o mesmo paciente pode ter mais de uma notificação (transferência entre unidades, reingresso). Marcamos
como **possível duplicidade** registros com o mesmo ano de nascimento, sexo, raça, município de residência e
data de diagnóstico. Eles **não** são removidos: os indicadores oficiais usam casos novos, que já excluem
transferências; e o notebook 10 mostra o efeito de removê-los.

In [6]:
chave = ['ANO_NASC', 'CS_SEXO', 'CS_RACA', 'ID_MN_RESI', 'DT_DIAG']
df['possivel_duplicata'] = raw.duplicated(subset=chave, keep=False) & raw['DT_DIAG'].notna()
df['duplicata_excedente'] = raw.duplicated(subset=chave, keep='first') & raw['DT_DIAG'].notna()

no_periodo = df['ano_diag'].between(2014, 2024)
ii = no_periodo & df['mun_res'].isin([tb.COD_ILHEUS, tb.COD_ITABUNA])
qualidade = pd.DataFrame({
    'Ilhéus + Itabuna (residentes)': [ii.sum(), (df['possivel_duplicata'] & ii).sum(), (df['duplicata_excedente'] & ii).sum(),
                                      (df['erro_cronologico'] & ii).sum(), (df['data_trat_invalida'] & ii).sum()],
}, index=['Registros', 'Envolvidos em possível duplicidade', 'Registros excedentes (cópias)',
          'Encerramento antes do diagnóstico ou > 5 anos', 'Início do tratamento inválido (<0 ou >365 dias)'])
qualidade

,Ilhéus + Itabuna (residentes)
Registros,3251
Envolvidos em possível duplicidade,76
Registros excedentes (cópias),39
Encerramento antes do diagnóstico ou > 5 anos,1
Início do tratamento inválido (<0 ou >365 dias),33


In [7]:
cols_dup = ['ano_diag', 'cidade_res', 'idade', 'sexo', 'tipo_entrada', 'desfecho', 'arquivo']
(df[ii & df['possivel_duplicata']].sort_values(['dt_diag', 'idade'])[cols_dup].head(12))

,ano_diag,cidade_res,idade,sexo,tipo_entrada,desfecho,arquivo
31,2014,Ilhéus,52.0,Masculino,Caso novo,Abandono,TUBEBR14.csv
21,2014,Ilhéus,61.0,Masculino,Caso novo,Cura,TUBEBR14.csv
23145,2014,Itabuna,19.0,Feminino,Caso novo,Abandono,TUBEBR14.csv
23146,2014,Itabuna,19.0,Feminino,Reingresso após abandono,Abandono,TUBEBR14.csv
22934,2014,Itabuna,53.0,Masculino,Caso novo,Abandono,TUBEBR14.csv
22935,2014,Itabuna,53.0,Masculino,Reingresso após abandono,Óbito por outras causas,TUBEBR14.csv
22914,2014,Itabuna,22.0,Feminino,Caso novo,Abandono,TUBEBR14.csv
22915,2014,Itabuna,22.0,Feminino,Reingresso após abandono,Abandono,TUBEBR14.csv
42011,2015,Itabuna,27.0,Masculino,Caso novo,Abandono,TUBEBR15.csv
42012,2015,Itabuna,27.0,Masculino,Reingresso após abandono,Óbito por outras causas,TUBEBR15.csv


Muitos pares combinam um "caso novo" com um "reingresso após abandono" na mesma data de diagnóstico, o que
indica o mesmo paciente notificado duas vezes. São 39 cópias em 3.251 registros (1,2%), pouco para mudar as
conclusões, mas o efeito é checado nos notebooks seguintes.

## 6. Coorte para indicadores do programa

Para cura, abandono e óbito seguimos o Ministério da Saúde: **casos novos** (tipo de entrada = caso novo),
excluindo **mudança de diagnóstico**. Também marcamos a coorte de casos novos **pulmonares com confirmação
laboratorial**, que é o indicador pactuado de cura.

In [8]:
df['coorte_casos_novos'] = df['tipo_entrada'].eq('Caso novo') & df['desfecho'].ne('Mudança de diagnóstico') & no_periodo
df['coorte_pulmonar_lab'] = df['coorte_casos_novos'] & df['pulmonar'] & df['confirmacao_lab'].eq(1)
df['caso_novo'] = df['tipo_entrada'].eq('Caso novo')
df['retratamento'] = df['tipo_entrada'].isin(['Recidiva', 'Reingresso após abandono'])

# Residente x não residente (do ponto de vista do município que notificou)
df['residente_no_mun_notificacao'] = df['mun_res'] == df['mun_not']

resumo = (df[ii].groupby('cidade_res')
          .agg(registros=('ano_diag', 'size'), casos_novos=('caso_novo', 'sum'),
               coorte_casos_novos=('coorte_casos_novos', 'sum'), coorte_pulmonar_lab=('coorte_pulmonar_lab', 'sum')))
resumo

,registros,casos_novos,coorte_casos_novos,coorte_pulmonar_lab
cidade_res,,,,
Ilhéus,1489,1161,1161,712
Itabuna,1762,1482,1482,793


## 7. Exportação

In [9]:
tb.PROCESSED.mkdir(parents=True, exist_ok=True)
df = df[df['ano_diag'].between(2013, 2025)]
df.to_csv(tb.PROCESSED / 'tb_base_analitica.csv.gz', index=False, compression='gzip')

agregado['ano_diag'] = agregado['ano_diag'].astype('Int64')
agregado.to_csv(tb.PROCESSED / 'tb_brasil_agregado.csv.gz', index=False, compression='gzip')
print(f"tb_base_analitica.csv.gz: {len(df)} registros e {df.shape[1]} colunas")
print(f"tb_brasil_agregado.csv.gz: {len(agregado)} linhas")
print('Vitória da Conquista (residentes, 2014-2024):',
      int((df['mun_res'].eq('293330') & df['ano_diag'].between(2014, 2024)).sum()), 'registros')

tb_base_analitica.csv.gz: 287958 registros e 74 colunas
tb_brasil_agregado.csv.gz: 83290 linhas
Vitória da Conquista (residentes, 2014-2024): 756 registros


## Conclusões

- A base analítica tem todos os registros ligados à Bahia e às 10 cidades, decodificados com um único dicionário.
- Ilhéus e Itabuna passam a ser definidas por **residência** nas análises de incidência e perfil.
- A população anual usa os Censos 2010 e 2022, evitando o salto artificial das estimativas antigas.
- Vitória da Conquista agora aparece com o volume real de casos.